# Dijet data versus MC distributions

Compare unit-normalized nominal reconstructed-data, eta-dependent JER-default reconstructed-MC, and nominal generator-level-MC dijet distributions. The notebook draws pTave spectra for each configured jet-eta acceptance and pseudorapidity shapes in every interval from `TEST_DIJET_PTAVE_BINS`. The pTave normalization interval is specified in physical values and converted to ROOT bins with `FindBin(low + 0.001)` and `FindBin(high - 0.001)`.

## Environment and imports

This notebook locates the repository dynamically and imports PyROOT from the
active project environment. Start Jupyter from the repository root with
`py-env/bin/python -m jupyter notebook`; no machine-specific ROOT paths are
added at runtime.


In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os

import sys

# Locate the repository without relying on a machine-specific absolute path.
PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "CMakeLists.txt").is_file()
        and (candidate / "hist_analysis").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError(
        "Cannot locate the jetAnalysis repository. Start Jupyter from its root."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root

# Batch mode keeps plots reproducible and sends them to notebook/output files.
ROOT = load_root(batch=True)

from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import TEST_DIJET_PTAVE_BINS
from hist_analysis.python.histogram_io import (
    load_histogram, resolve_combined_file, resolve_data_file,
)
from hist_analysis.python.histogram_ops import normalize_histogram
from hist_analysis.python.plotting import draw_closure
from hist_analysis.python.projections import project_semantic_th2
from hist_analysis.python.root_style import (
    DEFAULT_PLOT_STYLE, draw_text_block, save_canvas, set_1d_style,
    set_legend_style, set_pad_style, style_single_panel_axes,
)


In [ ]:
ROOT.gStyle.SetOptStat(0)
ROOT.TH1.AddDirectory(False)
ROOT.gStyle.SetPalette(ROOT.kBird)

## Configuration

`DATA_DIRECTION` selects `combined`, `Pbgoing`, or `pgoing` data for all four triggers. `DATA_SELECTION` selects the matching `jetId`, `trkMax`, or `noSel` production. Filenames are resolved through the shared data-output convention.

`NORMALIZATION_RANGE` is a half-open interval in GeV. For example, `(100.0, 120.0)` includes the ROOT bins found from 100.001 and 119.999 GeV. Pseudorapidity projections use the shared half-open intervals in `TEST_DIJET_PTAVE_BINS` and are independently normalized to unit integral. `PTAVE_RATIO_RANGE` and `ETA_RATIO_RANGE` configure the two ratio-panel ranges independently.

In [ ]:
GENERATOR = 'embedding'  # embedding or pythia
MC_FILE_STEM = 'jetId'
MC_FILE = resolve_combined_file(BASE_DIR, GENERATOR, MC_FILE_STEM)
DATA_DIR = Path(os.environ.get('PPB_DATA_DIR', BASE_DIR / 'exp'))
DATA_DIRECTION = 'combined'  # combined, Pbgoing, or pgoing
DATA_SELECTION = 'jetId'     # jetId, trkMax, or noSel
DATA_FILES = {
    trigger: resolve_data_file(DATA_DIR, trigger, DATA_DIRECTION, DATA_SELECTION)
    for trigger in ('mb', 'jet60', 'jet80', 'jet100')
}
SAMPLE_LABELS = {
    'mb': 'Minimum bias', 'jet60': 'Jet60',
    'jet80': 'Jet80', 'jet100': 'Jet100',
}
ETA_CUTS = (1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.5)
ETA_CUT_INDICES = tuple(range(len(ETA_CUTS)))
NORMALIZATION_RANGE = (110.0, 130.0)
PTAVE_DISPLAY_RANGE = (40.0, 500.0)
PTAVE_RATIO_RANGE = (0.75, 1.25)
ETA_RATIO_RANGE = (0.65, 1.15)
REBIN_PTAVE = 1
REBIN_ETA = 2
DRAW_GRID = True
LOG_Y = True
SAVE_PDF = True
SAVE_PNG = False
OUTPUT_DIR = Path(os.environ.get(
    'DIJET_DATA_VS_MC_OUTPUT_DIR',
    PROJECT_ROOT / 'hist_analysis' / 'output' / 'dijet_data_vs_mc',
))

if NORMALIZATION_RANGE[0] >= NORMALIZATION_RANGE[1]:
    raise ValueError('NORMALIZATION_RANGE must satisfy low < high')
if any(index < 0 or index >= len(ETA_CUTS) for index in ETA_CUT_INDICES):
    raise IndexError(f'Invalid ETA_CUT_INDICES: {ETA_CUT_INDICES}')
if any(isinstance(value, bool) or not isinstance(value, int) or value < 1
       for value in (REBIN_PTAVE, REBIN_ETA)):
    raise ValueError('REBIN_PTAVE and REBIN_ETA must be positive integers')
for filename in (MC_FILE, *DATA_FILES.values()):
    if not filename.exists():
        raise FileNotFoundError(f'Missing configured ROOT file: {filename}')

MC_FILE, DATA_FILES

## Projection, normalization, and drawing helpers

The MC-reco curve uses `hRecoDijetPtEtaCMJerDefExtra`, the default JER-smearing variation with eta-dependent scaling. MC gen and data use the nominal `hGenDijetPtEtaCM` and `hRecoDijetPtEtaCM` histograms, respectively. The three physics levels use shared style indices 0 (MC reco), 1 (MC gen), and 2 (data). Each pTave spectrum is normalized in `NORMALIZATION_RANGE` and compared with data; each pseudorapidity projection is normalized over its full in-range bin content and compared with nominal MC gen. Ratios use standard independent-error propagation.

In [ ]:
CURVES = (
    ('MC reco', MC_FILE, 'hRecoDijetPtEtaCMJerDefExtra_{eta_cut_index}', 0),
    ('MC gen', MC_FILE, 'hGenDijetPtEtaCM_{eta_cut_index}', 1),
    ('Data', None, 'hRecoDijetPtEtaCM_{eta_cut_index}', 2),
)

def _project_ptave(filename, key, name):
    source = load_histogram(str(filename), key)
    if not source.InheritsFrom('TH2'):
        raise TypeError(f'{key!r} in {filename} is not a TH2')
    projection = source.ProjectionX(
        name, 1, source.GetYaxis().GetNbins(), 'e',
    )
    projection.SetDirectory(0)
    if REBIN_PTAVE > 1:
        projection.Rebin(REBIN_PTAVE)
    return projection

def _project_eta(filename, key, ptave_range, name):
    source = load_histogram(str(filename), key)
    if not source.InheritsFrom('TH2'):
        raise TypeError(f'{key!r} in {filename} is not a TH2')
    projection = project_semantic_th2(
        source, 'eta', ptave_range, name=name,
    )
    if REBIN_ETA > 1:
        projection.Rebin(REBIN_ETA)
    return projection

def _normalize_in_value_range(histogram, low, high):
    axis = histogram.GetXaxis()
    bin_low = axis.FindBin(low + 0.001)
    bin_high = axis.FindBin(high - 0.001)
    integral = histogram.Integral(bin_low, bin_high)
    if integral <= 0.0:
        raise ValueError(
            f'Cannot normalize {histogram.GetName()}: integral over bins '
            f'{bin_low}--{bin_high} ({low:g}--{high:g} GeV) is {integral:g}'
        )
    histogram.Scale(1.0 / integral)
    return bin_low, bin_high, integral

def _assert_compatible_binning(reference, candidate):
    if reference.GetNbinsX() != candidate.GetNbinsX():
        raise ValueError('Cannot overlay histograms with different bin counts')
    for edge_index in range(1, reference.GetNbinsX() + 2):
        reference_edge = reference.GetXaxis().GetBinLowEdge(edge_index)
        candidate_edge = candidate.GetXaxis().GetBinLowEdge(edge_index)
        if abs(reference_edge - candidate_edge) > 1.0e-9:
            raise ValueError('Cannot overlay histograms with different bin edges')

def _draw_eta_cut(sample, eta_cut_index):
    eta_cut = ETA_CUTS[eta_cut_index]
    histograms = {}
    normalization_details = {}
    for label, configured_file, key_template, style_index in CURVES:
        filename = DATA_FILES[sample] if configured_file is None else configured_file
        key = key_template.format(eta_cut_index=eta_cut_index)
        name = f'h_{sample}_{label.lower().replace(" ", "_")}_eta_{eta_cut_index}'
        histogram = _project_ptave(filename, key, name)
        if histograms:
            _assert_compatible_binning(next(iter(histograms.values())), histogram)
        normalization_details[label] = _normalize_in_value_range(
            histogram, *NORMALIZATION_RANGE,
        )
        set_1d_style(histogram, style_index)
        style_single_panel_axes(histogram)
        histogram.SetTitle('')
        histogram.GetXaxis().SetTitle('p_{T}^{ave} (GeV)')
        # The projection is scaled to unit bin-content sum, not divided by bin width.
        histogram.GetYaxis().SetTitle('Fraction of dijets / bin')
        histogram.GetXaxis().SetRangeUser(*PTAVE_DISPLAY_RANGE)
        histograms[label] = histogram

    ratios = {}
    for label in ('MC reco', 'MC gen'):
        ratio = histograms[label].Clone(
            f'{histograms[label].GetName()}_over_data',
        )
        ratio.SetDirectory(0)
        ratio.Divide(histograms['Data'])
        ratio.GetYaxis().SetTitle('MC / Data')
        ratio.GetYaxis().SetRangeUser(*PTAVE_RATIO_RANGE)
        ratio.GetXaxis().SetRangeUser(*PTAVE_DISPLAY_RANGE)
        ratios[label] = ratio

    canvas_name = f'c_{sample}_ptave_eta_{eta_cut_index}'
    canvas = ROOT.TCanvas(
        canvas_name, '', DEFAULT_PLOT_STYLE.canvas_width,
        DEFAULT_PLOT_STYLE.canvas_height,
    )
    top_pad = ROOT.TPad('top_' + canvas_name, '', 0.0, 0.30, 1.0, 1.0)
    ratio_pad = ROOT.TPad('ratio_' + canvas_name, '', 0.0, 0.0, 1.0, 0.30)
    top_pad.SetLeftMargin(DEFAULT_PLOT_STYLE.ratio_left_margin)
    top_pad.SetRightMargin(DEFAULT_PLOT_STYLE.right_margin)
    top_pad.SetTopMargin(DEFAULT_PLOT_STYLE.top_margin)
    top_pad.SetBottomMargin(DEFAULT_PLOT_STYLE.ratio_top_bottom_margin)
    ratio_pad.SetLeftMargin(DEFAULT_PLOT_STYLE.ratio_left_margin)
    ratio_pad.SetRightMargin(DEFAULT_PLOT_STYLE.right_margin)
    ratio_pad.SetTopMargin(DEFAULT_PLOT_STYLE.ratio_bottom_top_margin)
    ratio_pad.SetBottomMargin(DEFAULT_PLOT_STYLE.ratio_bottom_margin)
    top_pad.SetGrid(DRAW_GRID, DRAW_GRID)
    ratio_pad.SetGrid(DRAW_GRID, DRAW_GRID)
    top_pad.SetLogy(LOG_Y)
    top_pad.Draw()
    ratio_pad.Draw()
    top_pad.cd()
    maximum = max(histogram.GetMaximum() for histogram in histograms.values())
    positive_bins = [
        histogram.GetBinContent(bin_index)
        for histogram in histograms.values()
        for bin_index in range(1, histogram.GetNbinsX() + 1)
        if histogram.GetBinContent(bin_index) > 0.0
    ]
    for curve_index, (label, histogram) in enumerate(histograms.items()):
        histogram.SetMaximum(maximum * (8.0 if LOG_Y else 1.5))
        if LOG_Y and positive_bins:
            histogram.SetMinimum(min(positive_bins) * 0.5)
        histogram.GetXaxis().SetLabelSize(0.0)
        histogram.GetXaxis().SetTitleSize(0.0)
        histogram.Draw('E1' if curve_index == 0 else 'E1 SAME')

    legend = ROOT.TLegend(0.62, 0.73, 0.88, 0.88)
    set_legend_style(legend)
    for label, histogram in histograms.items():
        legend.AddEntry(histogram, label, 'p')
    legend.Draw()
    text = draw_text_block(top_pad, (
        'pPb 8.16 TeV', SAMPLE_LABELS[sample],
        f'|#eta_{{CM}}^{{jet}}| < {eta_cut:g}',
        f'normalized in {NORMALIZATION_RANGE[0]:g} #leq p_{{T}}^{{ave}} < {NORMALIZATION_RANGE[1]:g} GeV',
    ), text_size=0.03)

    ratio_pad.cd()
    for ratio_index, ratio in enumerate(ratios.values()):
        ratio.GetXaxis().SetTitleSize(DEFAULT_PLOT_STYLE.ratio_axis_title_size)
        ratio.GetXaxis().SetLabelSize(DEFAULT_PLOT_STYLE.ratio_axis_label_size)
        ratio.GetXaxis().SetTitleOffset(DEFAULT_PLOT_STYLE.ratio_x_title_offset)
        ratio.GetYaxis().SetTitleSize(DEFAULT_PLOT_STYLE.ratio_axis_title_size)
        ratio.GetYaxis().SetLabelSize(DEFAULT_PLOT_STYLE.ratio_axis_label_size)
        ratio.GetYaxis().SetTitleOffset(DEFAULT_PLOT_STYLE.ratio_y_title_offset)
        ratio.GetYaxis().SetNdivisions(DEFAULT_PLOT_STYLE.ratio_axis_divisions)
        ratio.Draw('E1' if ratio_index == 0 else 'E1 SAME')
    unity = ROOT.TLine(PTAVE_DISPLAY_RANGE[0], 1.0, PTAVE_DISPLAY_RANGE[1], 1.0)
    unity.SetLineStyle(2)
    unity.Draw()

    canvas.cd()
    canvas.Modified()
    canvas.Update()
    canvas._dijet_data_vs_mc_objects = [
        top_pad, ratio_pad, legend, unity, *text,
        *histograms.values(), *ratios.values(),
    ]

    eta_tag = int(round(10.0 * eta_cut))
    if SAVE_PDF:
        save_canvas(
            canvas, OUTPUT_DIR / f'{sample}_dijet_ptave_etaCM_{eta_tag}.pdf',
            save_png=SAVE_PNG,
        )
    print(f'{sample}, eta cut {eta_cut:g}:', normalization_details)
    display(canvas)
    return {
        'histograms': histograms, 'ratios_to_data': ratios,
        'normalization': normalization_details,
        'canvas': canvas,
    }

def analyze_sample(sample):
    if sample not in DATA_FILES:
        raise KeyError(f'Unknown data sample: {sample!r}')
    return {
        eta_cut_index: _draw_eta_cut(sample, eta_cut_index)
        for eta_cut_index in ETA_CUT_INDICES
    }

def _draw_eta_distribution(sample, eta_cut_index, ptave_range):
    eta_cut = ETA_CUTS[eta_cut_index]
    low, high = ptave_range
    histograms = {}
    style_indices = {}
    for label, configured_file, key_template, style_index in CURVES:
        filename = DATA_FILES[sample] if configured_file is None else configured_file
        key = key_template.format(eta_cut_index=eta_cut_index)
        name = (f'h_{sample}_{label.lower().replace(" ", "_")}_etaCM_'
                f'{eta_cut_index}_ptave_{low:g}_{high:g}')
        histogram = _project_eta(filename, key, ptave_range, name)
        if histograms:
            _assert_compatible_binning(next(iter(histograms.values())), histogram)
        histograms[label] = normalize_histogram(histogram, 'integral')
        style_indices[label] = style_index

    eta_tag = int(round(10.0 * eta_cut))
    output = None
    if SAVE_PDF:
        output = OUTPUT_DIR / (
            f'{sample}_dijet_etaCM_{eta_tag}_ptave_{low:g}_{high:g}.pdf'
        )
    canvas, ratios = draw_closure(
        histograms, 'MC gen', title='',
        x_title='#eta_{CM}^{dijet}', y_title='Fraction of dijets / bin',
        ratio_range=ETA_RATIO_RANGE, log_y=False, grid=DRAW_GRID,
        output=output, save_png=SAVE_PNG, draw_nominal_ratio=False,
        canvas_name=f'c_{sample}_etaCM_{eta_tag}_ptave_{low:g}_{high:g}',
        annotations=(
            'pPb 8.16 TeV', SAMPLE_LABELS[sample],
            f'{low:g} #leq p_{{T}}^{{ave}} < {high:g} GeV',
            f'|#eta_{{CM}}^{{jet}}| < {eta_cut:g}',
        ),
        x_range=(-eta_cut - 0.1, eta_cut + 0.1),
        ratio_option='', style_indices=style_indices,
    )
    display(canvas)
    return {'histograms': histograms, 'ratios_to_gen': ratios, 'canvas': canvas}

def analyze_eta_sample(sample):
    if sample not in DATA_FILES:
        raise KeyError(f'Unknown data sample: {sample!r}')
    return {
        (eta_cut_index, ptave_range): _draw_eta_distribution(
            sample, eta_cut_index, ptave_range,
        )
        for eta_cut_index in ETA_CUT_INDICES
        for ptave_range in TEST_DIJET_PTAVE_BINS
    }

## Minimum bias

In [ ]:
mb_results = analyze_sample('mb')
mb_eta_results = analyze_eta_sample('mb')

## Jet60

In [ ]:
jet60_results = analyze_sample('jet60')
jet60_eta_results = analyze_eta_sample('jet60')

## Jet80

In [ ]:
jet80_results = analyze_sample('jet80')
jet80_eta_results = analyze_eta_sample('jet80')

## Jet100

In [ ]:
jet100_results = analyze_sample('jet100')
jet100_eta_results = analyze_eta_sample('jet100')